# Summarisation and ambient clinical documentation

Exercise notebook for the lesson **[Summarisation and ambient clinical documentation](https://ramkumarmd.com/medical-ai/generative-ai-llms/summarisation-ambient-documentation)** in the *Generative AI & LLMs* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Ambient AI scribes record a consultation, with consent, and draft a clinical note for the clinician to edit and sign; related tools summarise records and discharge information. In a randomised trial of 238 physicians, one scribe reduced time spent in notes by 9.5% and the other showed no significant change, with possible improvements in burnout measures for both. Clinicians reported occasional clinically significant inaccuracies, so consent, careful review and auditing are essential.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Auditing drafts: what do clinicians change?

Compare a synthetic AI draft with the clinician's final note, sentence by sentence.

In [ ]:
import difflib

draft = """Patient seen for diabetes review.
Reports good adherence to metformin.
No hypoglycaemia.
Foot examination normal.
Plan to start empagliflozin 10 mg daily.
Review in 3 months.""".split("\n")

final = """Patient seen for diabetes review.
Reports missing evening metformin doses about twice a week.
No hypoglycaemia.
Foot examination not performed today; booked for next visit.
Plan to start empagliflozin 10 mg daily after checking kidney function.
Review in 3 months.""".split("\n")

changed = 0
for d, f in zip(draft, final):
    similarity = difflib.SequenceMatcher(None, d, f).ratio()
    if similarity < 1.0:
        changed += 1
        print(f"EDITED ({similarity:.2f})\n  AI draft: {d}\n  Final:    {f}")
print(f"\n{changed} of {len(draft)} sentences edited by the clinician")

# EDITED (0.51)
#   AI draft: Reports good adherence to metformin.
#   Final:    Reports missing evening metformin doses about twice a week.
# EDITED (0.55)
#   AI draft: Foot examination normal.
#   Final:    Foot examination not performed today; booked for next visit.
# EDITED (0.72)
#   AI draft: Plan to start empagliflozin 10 mg daily.
#   Final:    Plan to start empagliflozin 10 mg daily after checking kidney function.
#
# 3 of 6 sentences edited by the clinician

## Exercise data

Synthetic data written for this exercise (no real patients). Run this cell before starting the steps.

In [ ]:
# Twenty synthetic consultations (no real patients): a short transcript summary, the AI draft and the
# clinician's final note, aligned sentence by sentence. "" means a sentence was removed (final) or added
# (draft). Each edited sentence carries a label: "style", "factual correction", "added safety information"
# or "removed fabricated content". Unedited sentences have label None.
def pair(transcript, rows):
    return {"transcript": transcript,
            "draft": [r[0] for r in rows], "final": [r[1] for r in rows], "labels": [r[2] for r in rows]}

S, FC, SAFE, FAB = "style", "factual correction", "added safety information", "removed fabricated content"

pairs = [
    pair("Diabetes review. HbA1c 7.9%. Takes metformin, misses evening dose twice a week. No hypos. Review 3 months.",
         [("Diabetes review.", "Diabetes review.", None),
          ("Good adherence to metformin.", "Misses evening metformin dose about twice a week.", FC),
          ("HbA1c 7.9%.", "HbA1c 7.9%.", None),
          ("Foot examination normal.", "", FAB),
          ("Review in 3 months.", "Review in 3 months.", None)]),
    pair("New type 2 diabetes. HbA1c 9.4%. Start metformin 500 mg, increase after 2 weeks. Explained stomach upset.",
         [("New diagnosis of type 2 diabetes, HbA1c 9.4%.", "New diagnosis of type 2 diabetes, HbA1c 9.4%.", None),
          ("Metformin 500 mg started.", "Metformin 500 mg started, to increase after 2 weeks.", FC),
          ("Pt counselled re side effects.", "Counselled about possible stomach upset.", S)]),
    pair("Type 1 diabetes. Night hypos 3 times a week. Reduce basal insulin by 10%. Talked about glucagon.",
         [("Nocturnal hypoglycaemia three times weekly.", "Nocturnal hypoglycaemia three times weekly.", None),
          ("Basal insulin reduced by 10%.", "Basal insulin reduced by 10%.", None),
          ("", "Glucagon kit checked and in date; family shown how to use it.", SAFE)]),
    pair("CKD 3a, eGFR 52. Start empagliflozin 10 mg. Recheck kidney function in 4 weeks.",
         [("eGFR 52 (CKD 3a).", "eGFR 52 (CKD 3a).", None),
          ("Empagliflozin 10 mg daily started.", "Empagliflozin 10 mg daily started.", None),
          ("", "Advised to pause empagliflozin if unwell and unable to eat or drink (sick-day rules).", SAFE),
          ("Kidney function in 4 weeks.", "Recheck kidney function in 4 weeks.", S)]),
    pair("Tingling feet. Monofilament reduced both feet. Refer podiatry.",
         [("Tingling in both feet.", "Tingling in both feet.", None),
          ("Monofilament reduced bilaterally.", "Monofilament reduced bilaterally.", None),
          ("Pedal pulses palpable.", "", FAB),
          ("Referred to podiatry.", "Referred to podiatry.", None)]),
    pair("Gestational diabetes, 28 weeks. Fasting 5.8. Diet advice. Review 1 week.",
         [("28 weeks pregnant, gestational diabetes.", "28 weeks pregnant, gestational diabetes.", None),
          ("Fasting glucose 8.5 mmol/L.", "Fasting glucose 5.8 mmol/L.", FC),
          ("Dietary advice given.", "Dietary advice given.", None),
          ("Review in 1 week.", "Review in 1 week.", None)]),
    pair("Weight up. HbA1c 8.1%. On gliclazide. Stop gliclazide, start semaglutide 0.25 mg weekly.",
         [("Weight gain noted.", "Weight gain noted.", None),
          ("HbA1c 8.1%.", "HbA1c 8.1%.", None),
          ("Gliclazide continued; semaglutide started.", "Gliclazide stopped; semaglutide 0.25 mg weekly started.", FC),
          ("Advised on diet and exercise for 30 minutes daily.", "", FAB)]),
    pair("Blood pressure 152/94 on ramipril. Increase to 10 mg. Recheck 2 weeks.",
         [("BP 152/94 on ramipril.", "Blood pressure 152/94 on ramipril.", S),
          ("Ramipril increased to 10 mg.", "Ramipril increased to 10 mg.", None),
          ("", "Blood test for kidney function and potassium in 1-2 weeks.", SAFE),
          ("Recheck blood pressure in 2 weeks.", "Recheck blood pressure in 2 weeks.", None)]),
    pair("Frail, 81. Two falls. HbA1c 6.0% on gliclazide. Stop gliclazide.",
         [("81-year-old, frail, two recent falls.", "81-year-old, frail, two recent falls.", None),
          ("HbA1c 6.0% on gliclazide.", "HbA1c 6.0% on gliclazide.", None),
          ("Gliclazide stopped.", "Gliclazide stopped.", None),
          ("Cardiovascular examination unremarkable.", "", FAB)]),
    pair("Retinal screening shows background retinopathy. HbA1c 8.6%. Rescreen 12 months.",
         [("Background retinopathy on screening.", "Background diabetic retinopathy on screening.", S),
          ("HbA1c 8.6%.", "HbA1c 8.6%.", None),
          ("Rescreen in 6 months.", "Rescreen in 12 months.", FC),
          ("", "Told to seek same-day care for sudden vision loss or new floaters.", SAFE)]),
    pair("Foot ulcer right big toe, 1 cm, no infection. Offloading. Podiatry within a week.",
         [("1 cm ulcer, right hallux, no signs of infection.", "1 cm ulcer, right hallux, no signs of infection.", None),
          ("Offloading advised.", "Offloading advised.", None),
          ("Podiatry review within 1 week.", "Podiatry review within 1 week.", None),
          ("", "Advised to attend urgently if redness, swelling, fever or a bad smell develops.", SAFE)]),
    pair("Chest tightness on exertion 2 weeks. ECG normal. Urgent cardiology referral.",
         [("Exertional chest tightness for 2 weeks.", "Exertional chest tightness for 2 weeks.", None),
          ("ECG normal.", "ECG today normal.", S),
          ("Urgent cardiology referral.", "Urgent cardiology referral.", None),
          ("", "Told to call emergency services if pain at rest or lasting more than 15 minutes.", SAFE)]),
    pair("HbA1c 7.2%. Stable. Same treatment. Review 6 months.",
         [("HbA1c 7.2%, stable.", "HbA1c 7.2%, stable.", None),
          ("Continue current treatment.", "Continue current treatment.", None),
          ("Review in 6 months.", "Review in 6 months.", None)]),
    pair("Type 1 diabetes, starting insulin pump. Pump training booked.",
         [("Starting insulin pump therapy.", "Starting insulin pump therapy.", None),
          ("Pump training booked.", "Pump training booked.", None),
          ("Injection sites examined, no lipohypertrophy.", "", FAB)]),
    pair("eGFR 38 on metformin 1 g twice daily. Reduce to 500 mg twice daily.",
         [("eGFR 38.", "eGFR 38.", None),
          ("Metformin reduced to 500 mg once daily.", "Metformin reduced to 500 mg twice daily.", FC),
          ("Recheck eGFR in 3 months.", "Recheck eGFR in 3 months.", None)]),
    pair("LDL 3.6, not on statin. Discussed; agreed atorvastatin 20 mg.",
         [("LDL 3.6 mmol/L, not on a statin.", "LDL 3.6 mmol/L, not on a statin.", None),
          ("Atorvastatin 20 mg started.", "Atorvastatin 20 mg started after discussion of benefits and side effects.", S),
          ("Lipids repeated today.", "", FAB)]),
    pair("Recent DKA admission after missed insulin. HbA1c 11.0%. Sick-day rules. Psychology referral.",
         [("Recent DKA after missed insulin doses.", "Recent DKA after missed insulin doses.", None),
          ("HbA1c 11.0%.", "HbA1c 11.0%.", None),
          ("Sick-day rules discussed.", "Sick-day rules discussed; written copy given.", S),
          ("Psychology referral.", "Psychology referral made.", S)]),
    pair("Tired. Haemoglobin 10.6. Check iron and B12. On metformin.",
         [("Tiredness.", "Tiredness.", None),
          ("Haemoglobin 10.6.", "Haemoglobin 10.6 g/dL.", S),
          ("Iron studies and B12 requested.", "Iron studies and B12 requested.", None),
          ("Advised to avoid driving until results are back.", "", FAB)]),
    pair("Ramadan in 3 weeks. On metformin and gliclazide. Plan for fasting.",
         [("Planning to fast during Ramadan.", "Planning to fast during Ramadan.", None),
          ("Metformin to be taken at the evening meal.", "Metformin doses moved to the pre-dawn and evening meals.", FC),
          ("", "Gliclazide dose reduced for fasting days; break the fast if glucose below 4 mmol/L.", SAFE)]),
    pair("Annual review. HbA1c 6.8%. BP 128/78. Feet checked: normal. Review 12 months.",
         [("Annual diabetes review.", "Annual diabetes review.", None),
          ("HbA1c 6.8%; blood pressure 128/78.", "HbA1c 6.8%; blood pressure 128/78.", None),
          ("Foot check normal.", "Foot check normal.", None),
          ("Review in 12 months.", "Review in 12 months.", None)]),
]
print(f"{len(pairs)} synthetic draft/final pairs")

## Exercise

**Goal:** Audit synthetic AI-drafted notes against their final versions and design a monitoring report.

**Data:** Twenty synthetic draft and final note pairs, written for the notebook (Created for this course; no real patient data)

### Step 1

Compute the share of sentences edited in each note pair with difflib.

In [ ]:
# Your code here


### Step 2

Classify each edit as style, factual correction, added safety information or removed fabricated content.

In [ ]:
# Your code here


### Step 3

Identify drafts that documented examinations or advice not present in the transcript provided.

In [ ]:
# Your code here


### Step 4

Produce a one-page monthly audit summary with three metrics a governance committee could track.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/generative-ai-llms/summarisation-ambient-documentation](https://ramkumarmd.com/medical-ai/generative-ai-llms/summarisation-ambient-documentation). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.